In [6]:
import os
import pandas as pd


In [8]:

PROJECT_ROOT = r"C:\Users\abc\projects\Cancer Cell Classification Project"

META_DIR = os.path.join(PROJECT_ROOT, "Data", "metadata")
IMG_DIR  = os.path.join(PROJECT_ROOT, "Data", "images")

image_csv_path = os.path.join(META_DIR, "BBBC021_v1_image.csv")
moa_csv_path   = os.path.join(META_DIR, "BBBC021_v1_moa.csv")

print("META_DIR:", META_DIR)
print("IMG_DIR:", IMG_DIR)
print("image_csv_path exists:", os.path.exists(image_csv_path))
print("moa_csv_path exists:", os.path.exists(moa_csv_path))


META_DIR: C:\Users\abc\projects\Cancer Cell Classification Project\Data\metadata
IMG_DIR: C:\Users\abc\projects\Cancer Cell Classification Project\Data\images
image_csv_path exists: True
moa_csv_path exists: True


In [10]:
img_df = pd.read_csv(image_csv_path)
moa_df = pd.read_csv(moa_csv_path)

print("img_df shape:", img_df.shape)
print("moa_df shape:", moa_df.shape)

display(img_df.head())
display(moa_df.head())


img_df shape: (13200, 13)
moa_df shape: (104, 3)


,TableNumber,ImageNumber,Image_FileName_DAPI,Image_PathName_DAPI,Image_FileName_Tubulin,Image_PathName_Tubulin,Image_FileName_Actin,Image_PathName_Actin,Image_Metadata_Plate_DAPI,Image_Metadata_Well_DAPI,Replicate,Image_Metadata_Compound,Image_Metadata_Concentration
0,4,233,G10_s1_w1BEDC2073-A983-4B98-95E9-84466707A25D.tif,Week4/Week4_27481,G10_s1_w2DCEC82F3-05F7-4F2F-B779-C5DF9698141E.tif,Week4/Week4_27481,G10_s1_w43CD51CBC-2370-471F-BA01-EE250B14B3C8.tif,Week4/Week4_27481,Week4_27481,G10,1,5-fluorouracil,0.003
1,4,234,G10_s2_w11C3B9BCC-E48F-4C2F-9D31-8F46D8B5B972.tif,Week4/Week4_27481,G10_s2_w2570437EF-C8DC-4074-8D63-7FA3A7271FEE.tif,Week4/Week4_27481,G10_s2_w400B21F33-BDAB-4363-92C2-F4FB7545F08C.tif,Week4/Week4_27481,Week4_27481,G10,1,5-fluorouracil,0.003
2,4,235,G10_s3_w1F4FCE330-C71C-4CA3-9815-EAF9B9876EB5.tif,Week4/Week4_27481,G10_s3_w2194A9AC7-369B-4D84-99C0-DA809B0042B8.tif,Week4/Week4_27481,G10_s3_w4E0452054-9FC1-41AB-8C5B-D0ACD058991F.tif,Week4/Week4_27481,Week4_27481,G10,1,5-fluorouracil,0.003
3,4,236,G10_s4_w1747818B4-FFA7-40EE-B0A0-6A5974AF2644.tif,Week4/Week4_27481,G10_s4_w298D4652F-B5BF-49F2-BE51-8149DF83EAFD.tif,Week4/Week4_27481,G10_s4_w42648D36D-6B77-41CD-B520-6E4C533D9ABC.tif,Week4/Week4_27481,Week4_27481,G10,1,5-fluorouracil,0.003
4,4,473,G10_s1_w10034568D-CC12-43C3-93A9-DC3782099DD3.tif,Week4/Week4_27521,G10_s1_w2A29ED14B-952C-4BA1-89B9-4F92B6DADEB4.tif,Week4/Week4_27521,G10_s1_w4DAA2E9D1-F6E9-45FA-ADC0-D341B647A680.tif,Week4/Week4_27521,Week4_27521,G10,2,5-fluorouracil,0.003


,compound,concentration,moa
0,PP-2,3.0,Epithelial
1,emetine,0.3,Protein synthesis
2,AZ258,1.0,Aurora kinase inhibitors
3,cytochalasin B,10.0,Actin disruptors
4,ALLN,3.0,Protein degradation


In [12]:
print("img_df columns:\n", img_df.columns.tolist())
print("\nmoa_df columns:\n", moa_df.columns.tolist())


img_df columns:
 ['TableNumber', 'ImageNumber', 'Image_FileName_DAPI', 'Image_PathName_DAPI', 'Image_FileName_Tubulin', 'Image_PathName_Tubulin', 'Image_FileName_Actin', 'Image_PathName_Actin', 'Image_Metadata_Plate_DAPI', 'Image_Metadata_Well_DAPI', 'Replicate', 'Image_Metadata_Compound', 'Image_Metadata_Concentration']

moa_df columns:
 ['compound', 'concentration', 'moa']


In [14]:
week_candidates = []
for col in img_df.columns:
    sample = img_df[col].astype(str).head(8000)
    if sample.str.contains("Week1_", case=False, na=False).any():
        week_candidates.append(col)

print("Columns containing 'Week1_':", week_candidates)


Columns containing 'Week1_': ['Image_FileName_DAPI', 'Image_PathName_DAPI', 'Image_FileName_Tubulin', 'Image_PathName_Tubulin', 'Image_FileName_Actin', 'Image_PathName_Actin', 'Image_Metadata_Plate_DAPI']


In [16]:
PLATE_COL = "Image_Metadata_Plate_DAPI"

plates_to_use = {"Week1_22361", "Week1_22381", "Week2_24121"}
subset_img_df = img_df[img_df[PLATE_COL].astype(str).isin(plates_to_use)].copy()

print("subset_img_df shape:", subset_img_df.shape)
display(subset_img_df[[PLATE_COL]].drop_duplicates())



subset_img_df shape: (720, 13)


,Image_Metadata_Plate_DAPI
384,Week2_24121
2204,Week1_22361
2208,Week1_22381


In [18]:
def find_cols(df, keywords):
    hits = []
    for c in df.columns:
        name = c.lower()
        if any(k in name for k in keywords):
            hits.append(c)
    return hits

print("Likely compound cols in subset_img_df:", find_cols(subset_img_df, ["compound"]))
print("Likely concentration cols in subset_img_df:", find_cols(subset_img_df, ["conc", "concentration", "dose"]))

print("\nLikely compound cols in moa_df:", find_cols(moa_df, ["compound"]))
print("Likely concentration cols in moa_df:", find_cols(moa_df, ["conc", "concentration", "dose"]))
print("Likely MoA label cols in moa_df:", find_cols(moa_df, ["moa", "mechanism"]))


Likely compound cols in subset_img_df: ['Image_Metadata_Compound']
Likely concentration cols in subset_img_df: ['Image_Metadata_Concentration']

Likely compound cols in moa_df: ['compound']
Likely concentration cols in moa_df: ['concentration']
Likely MoA label cols in moa_df: ['moa']


In [20]:
IMG_COMPOUND_COL = find_cols(subset_img_df, ["compound"])[0]
IMG_CONC_COL     = find_cols(subset_img_df, ["conc", "concentration", "dose"])[0]

MOA_COMPOUND_COL = find_cols(moa_df, ["compound"])[0]
MOA_CONC_COL     = find_cols(moa_df, ["conc", "concentration", "dose"])[0]
MOA_LABEL_COL    = find_cols(moa_df, ["moa", "mechanism"])[0]

# Standardise join keys
subset_img_df[IMG_COMPOUND_COL] = subset_img_df[IMG_COMPOUND_COL].astype(str).str.strip()
subset_img_df[IMG_CONC_COL]     = subset_img_df[IMG_CONC_COL].astype(str).str.strip()

moa_df[MOA_COMPOUND_COL] = moa_df[MOA_COMPOUND_COL].astype(str).str.strip()
moa_df[MOA_CONC_COL]     = moa_df[MOA_CONC_COL].astype(str).str.strip()

labeled_df = subset_img_df.merge(
    moa_df[[MOA_COMPOUND_COL, MOA_CONC_COL, MOA_LABEL_COL]],
    left_on=[IMG_COMPOUND_COL, IMG_CONC_COL],
    right_on=[MOA_COMPOUND_COL, MOA_CONC_COL],
    how="inner"
)
labeled_df = labeled_df[labeled_df[MOA_LABEL_COL] != "DMSO"]

print("labeled_df shape:", labeled_df.shape)

moa_counts = labeled_df[MOA_LABEL_COL].value_counts()
print("\nMoA counts in your selected plates:\n", moa_counts)

print("\nNumber of MoA groups present:", labeled_df[MOA_LABEL_COL].nunique())
print("\nMoA group names present:\n", sorted(labeled_df[MOA_LABEL_COL].unique()))

labeled_df shape: (136, 16)

MoA counts in your selected plates:
 moa
Microtubule stabilizers      96
Actin disruptors             20
Microtubule destabilizers     8
Protein degradation           8
DNA replication               4
Name: count, dtype: int64

Number of MoA groups present: 5

MoA group names present:
 ['Actin disruptors', 'DNA replication', 'Microtubule destabilizers', 'Microtubule stabilizers', 'Protein degradation']


In [22]:
print("Total in selected plates:", subset_img_df.shape[0])

# how many match MoA BEFORE removing DMSO
temp = subset_img_df.merge(
    moa_df,
    left_on=["Image_Metadata_Compound","Image_Metadata_Concentration"],
    right_on=["compound","concentration"],
    how="inner"
)
print("Matched to MoA table (including DMSO if present):", temp.shape[0])
print("MoA counts including DMSO:\n", temp["moa"].value_counts().head(15))

Total in selected plates: 720
Matched to MoA table (including DMSO if present): 208
MoA counts including DMSO:
 moa
Microtubule stabilizers      96
DMSO                         72
Actin disruptors             20
Microtubule destabilizers     8
Protein degradation           8
DNA replication               4
Name: count, dtype: int64
